# Algorithm Design — The Development Story

This notebook records the research process: the dead ends, design decisions, and
directions that led to the FairRank algorithm family. The complete mathematical
derivations, implementation walkthroughs, and a reference section for every variant
are in `algorithm_reference.ipynb`.

**Development history**

1. The journey to KNNFairRank
2. From a single fair vote to the multi-rank scheme — v1 and v2

---
### 1.3 The journey to KNNFairRank

#### First direction: pick a better global k

The simplest reaction to surface bias is to blame the choice of $k$. With a large $k$ the
neighbourhood swells and gets flooded with majority points simply because there are more of
them. The first attempt is therefore `KNNOptK`: select $k$ by cross-validation to find the
value that works best across the dataset as a whole. It helps — $k=1$ is chosen in ~62% of
folds — but the IR-vs-performance scatter (phase 1 §3.1) shows performance still degrades
with IR. A globally optimal $k$ is not enough.

#### Second direction: adapt k per query point

If a *global* $k$ is too coarse, perhaps $k$ should depend on the local structure around each
query point. We explored this with `KNNAdaptiveTopo` (persistent homology), as well as
entropy- and eigenvalue-based variants. These do beat `KNNOptK` modestly, but the gap to the
fair-rank family is large — which suggests local $k$ adaptation, while helpful, is not the
right frame.

#### The structural insight: the problem is not k

The reason neither a globally optimal nor a locally adaptive $k$ fully solves the problem is
that the issue is not *how many* neighbours we look at — it is *how we compare* the two
classes once we have the neighbours. Even with $k=1$ (the most local possible choice),
standard KNN is making a fundamentally unfair comparison.

To see why, we need to be precise about what "rank" means here.

**What rank means.** When KNN classifies a query point $x$, it computes the distance from
$x$ to every training point and sorts them from closest to furthest — separately for each
class:

- **Minority rank 1** — the single closest minority training point to $x$; distance $d_1^\text{min}(x)$
- **Minority rank 2** — the second closest minority training point; distance $d_2^\text{min}(x)$
- **Majority rank 1** — the single closest majority training point; distance $d_1^\text{maj}(x)$
- **Majority rank 2** — the second closest majority training point

Rank $k$ is simply position $k$ in the distance-sorted list for that class.

**The unfair comparison.** Standard KNN with $k=1$ asks: *is the closest minority point closer
than the closest majority point?* — i.e. does $d_1^\text{min}(x) < d_1^\text{maj}(x)$? This
is **minority rank-1 vs majority rank-1**, which looks symmetric but is not. Majority training
points are denser in space because there are simply more of them spread across the same region.
So majority rank-1 tends to be close to any query point by sheer density, while minority rank-1
tends to be further away for the same reason. No matter how well we adapt $k$, as long as we
compare the same rank across both classes we are comparing different things.

**The fix in one line.** A fair comparison would pair minority rank-1 against the majority
rank whose *expected* distance equals minority rank-1's expected distance. That majority rank
turns out to be exactly $r$ — the imbalance ratio. So instead of asking "is the closest
minority point closer than the **1st** closest majority point?", we ask "is the closest
minority point closer than the **$r$-th** closest majority point?". Section 2 derives this
result rigorously from a Poisson model and §2.5 extends it to multiple votes for variance
reduction.


---
---
## 2. From a Single Fair Vote to the Multi-Rank Scheme

The current algorithm (v3) is the third iteration. The previous two are not failures to
catalogue but a useful warm-up: they isolate the two ingredients of the final scheme — the
fair rank correction and the multi-rank averaging — and show what each one contributes on its
own.

### 3.1 v1 — a single fair vote

v1 was the simplest possible incarnation of the fair-rank idea: cast exactly one comparison,
$d_1^\text{min}(x) < d_{k_\text{eff}}^\text{maj}(x)$, and predict minority iff that single
comparison favours minority. The structural problem with v1 is not the single-vote part — that
becomes naturally noisy and §2.5 explains how to fix it. The problem with v1 was its choice of
$k_\text{eff}$: it used $r^{1/d}$, the *distance* ratio, where it should have used $r$, the
*rank* correction. §2.4 walks through why those two are different.

### 3.2 v2 — multiple votes with the wrong correction

v2 kept v1's $k_\text{eff} = r^{1/d}$ but added the multi-rank voting machinery from §2.5.
This is informative: it shows that aggregating votes does not rescue a wrong correction.
Whatever stability multi-vote averaging buys, it cannot undo a systematic under-correction in
the comparison rank — at high IR the under-correction is a factor of $r^{1 - 1/d}$, which
grows with $r$ and is severe in the low-dimensional tabular regime where most of our datasets
live.